# Train custom **ball** detector (YOLOv8n) for the Tria QCS6490
**Web Colab:** Runtime - Change runtime type - **T4 GPU**, then Runtime - Run all.

**Before running:** upload **ball_dataset.zip** to your Google Drive **My Drive** (root).


In [ ]:
!pip -q install ultralytics


## 1. Mount Drive + unzip the dataset
Approve the auth prompt. Expects ball_dataset.zip in My Drive root.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, os
src = '/content/drive/MyDrive/ball_dataset.zip'
assert os.path.exists(src), 'Not found: ' + src + '  -- put ball_dataset.zip in My Drive root'
zipfile.ZipFile(src).extractall('/content')
print('dataset:', os.listdir('/content/dataset'))


## 2. Point data.yaml at the Colab path


In [ ]:
yaml_text = """path: /content/dataset
train: train.txt
val: val.txt
nc: 1
names: [ball]
"""
open('/content/dataset/data.yaml', 'w').write(yaml_text)
print(yaml_text)


## 3. Train
imgsz=640 matches the board model input. ~10-20 min on a T4.


In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/dataset/data.yaml', epochs=100, imgsz=640,
            batch=16, patience=30, name='ball')


## 4. Validate


In [ ]:
m = model.val()
print('mAP50-95:', round(float(m.box.map), 3), ' mAP50:', round(float(m.box.map50), 3))


## 5. Export + save to Drive
Writes ball_best.pt (for Qualcomm AI Hub) + ball_best.onnx to My Drive.


In [ ]:
best = '/content/runs/detect/ball/weights/best.pt'
YOLO(best).export(format='onnx', imgsz=640, opset=12)
import shutil
shutil.copy(best, '/content/drive/MyDrive/ball_best.pt')
shutil.copy('/content/runs/detect/ball/weights/best.onnx', '/content/drive/MyDrive/ball_best.onnx')
print('Saved ball_best.pt + ball_best.onnx to your Google Drive (My Drive)')
